### Análisis Exploratorio de Datos (EDA)

#### Reto: ¿Qué canciones debeben estar en nuestras playlist premium y por que?

##### Carga de librerías e inicialización

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Carga de dataset limpio
spotify = pd.read_csv("../data/processed/spotify_2000_clean.csv")

# Verificación estructura de datos
spotify.info()

*Verificación del dataset*

In [ ]:
# Verificación tamaño dataset y valores faltantes
print("Tamaño de dataset:", spotify.shape)
print("\n Número de valores ausentes:")
print(spotify.isnull().sum())
print("\n Duplicados:", spotify.duplicated().sum())

# Muestra de los datos
display(spotify.head())

# Resumen estadístico de las variables acústicas y popularidad
acoustic_features = [
    'beats_per_minute_bpm', 'energy', 'danceability', 'loudness_db', 
    'liveness', 'valence', 'length_duration', 'acousticness', 'speechiness', 'popularity'
]

print("\nResumen estadístico de las variables acústicas y popularidad:")
spotify[acoustic_features].describe().round(1)


El dataset no presenta valores ausentes ni duplicados, las columnas están normalizadas y los datos en los tipos correctos.

#### ¿Cuáles son las canciones más populares?

*Distribución de popularidad*

In [ ]:
#Histograma de popularidad general del catálogo
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
sns.histplot(data=spotify['popularity'], kde=True, color='#1DB954', bins=30)
plt.title('Distribución de Popularity en el catálogo', fontsize=12, fontweight='bold') 
plt.xlabel('Puntuación de Popularidad')
plt.ylabel('Volumen de canciones')

# Boxplot popularidad
plt.subplot(1, 2, 2)
sns.boxplot(data=spotify['popularity'], color='gray', 
            flierprops={"markerfacecolor":"blue", "marker":"o"})
plt.title('Boxplot de Diagnóstico: Popularity', fontsize=12, fontweight='bold')
plt.xlabel('Puntuación de Popularidad')

plt.tight_layout()
plt.show()

Los valores de popularidad en todo el catálogo tienen como media general 59.4 puntos y una mediana de 62, con algunos outliers de valores bajos.

*Identificación de segmentos de alto y bajo rendimiento*

In [ ]:
percentil_75 = spotify['popularity'].quantile(0.75) 
# Umbral de bajo rendimiento
percentil_25 = spotify['popularity'].quantile(0.25) 

print(f"Corte alto rendimiento (Q3): >= {percentil_75}")
print(f"Corte bajo rendimiento (Q1): <= {percentil_25}\n")


El 25% de las pistas (Q3) tienen un score de popularidad mayor o igual a 71.0 y las pistas con una popularidad menor o igual 49.25 hacen parte del segmento de bajo rendimiento (Q1).

*Segmentación del catálogo por popularidad*

In [ ]:
# Calcula la popularidad por cuartiles
spotify["popularity_quartile"] = pd.qcut(
    spotify["popularity"],
    q=4,
    labels=[
        "q1_bajo",
        "q2_medio_bajo",
        "q3_medio_alto",
        "q4_alto"
    ],
    duplicates="drop"
)

# Agrupa datos por cuartil y calcula límites y promedios
quartile_summary = (
    spotify.groupby("popularity_quartile", observed=False)
    .agg(
        total_canciones=("title", "count"),
        popularity_min=("popularity", "min"),
        popularity_max=("popularity", "max"),
        popularity_promedio=("popularity", "mean"),
        energy_promedio=("energy", "mean"),
        danceability_promedio=("danceability", "mean"),
        acousticness_promedio=("acousticness", "mean"),
        valence_promedio=("valence", "mean"),
        loudness_promedio=("loudness_db", "mean")
    )
    .reset_index()
)
# Resumen
quartile_summary

El catálogo se encuentra distribuido de forma balanceada, el segmento de mayor popularidad se define a partir de un score de popularidad de 72 y el de menor rendimiento por debajo de 49.

**Insight de negocio**: el cuartil de mayor popularidad es la base óptima para la conformación de las listas premium. En cambio el de menor rendimiento puede no ser considerado para las siguientes elecciones.

#### 

##### ¿Qué géneros vale la pena incluir en las listas premium?


*Identificación de los géneros más populares*

In [ ]:
# Agrupación de géneros por popularidad
genre_popularity = spotify.groupby('top_genre').agg(
    total_tracks=('popularity', 'count'),
    popularity_mean=('popularity', 'mean')
).sort_values(by='popularity_mean', ascending=False).head(20)

# Gráfico del top 20 de los géneros más populares del catálogo
plt.figure(figsize=(10, 6))
sns.barplot(data=genre_popularity.reset_index(), x='popularity_mean', y='top_genre', color='#1DB954')
plt.title('Top 20 Géneros con mayor popularidad promedio', fontsize=14, fontweight='bold')
plt.xlabel('Popularidad Promedio')
plt.ylabel('Género')
plt.show()

El top 20 de los géneros con mayor popularidad promedio está liderado por 'celtic pukn' e 'indie pop', el resto del grupo presenta scores entre los 70 y 80 puntos. 

In [ ]:
# Incluir el volumen de canciones en cada género del top 20
# Configurar el tamaño y el estilo
plt.figure(figsize=(12, 6))
sns.set_theme(style="whitegrid")

# Resetear el índice para que 'top_genre' sea una columna accesible
df_plot = genre_popularity.reset_index()

# Crear el gráfico de barras 
ax = sns.barplot(
    data=df_plot, 
    x='popularity_mean', 
    y='top_genre', 
    color='#1DB954'
)
    
# Dibujar el recuento utilizando la posición de las filas del DataFrame
max_popularidad = df_plot['popularity_mean'].max()

for i, fila in df_plot.iterrows():
    popularidad = fila['popularity_mean']
    conteo_canciones = int(fila['total_tracks'])
    
    # Colocar el texto usando la coordenada del eje Y (que coincide con el índice i)
    ax.text(
        x=popularidad + (max_popularidad * 0.01), # Posición X (justo después del fin de la barra)
        y=i,                                      # Posición Y (el centro de la barra actual)
        s=f'{conteo_canciones}',                # Texto a mostrar (ej: n=15)
        va='center', 
        ha='left', 
        fontsize=10, 
        fontweight='bold',
        color='#333333'
    )

# Personalización del gráfico
ax.set_title('Top Géneros con mayor popularidad promedio y volumen de canciones', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Popularidad Promedio', fontsize=12)
ax.set_ylabel('Género', fontsize=12)

# Ajustar el límite de X para dar espacio al texto
ax.set_xlim(0, max_popularidad * 1.12)

plt.tight_layout()
plt.show()

El volumen de pistas de cada género fluctúa entre 1 y 7.

*Análisis de distribución de volumen por género*


In [ ]:
# Número de géneros existentes
print('Número de géneros existentes:')
print(spotify['top_genre'].nunique())


# Calcula el volumen de cada género
genre_count = spotify['top_genre'].value_counts()

# Grafica el histograma de frecuencias de los tamaños
plt.figure(figsize=(8, 4))

sns.histplot(
    x=genre_count.values, 
    kde=True, 
    color='green' , 
    bins=25
)

plt.title('Distribución del volumen por género', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Número de canciones')
plt.ylabel('Número de Géneros de ese tamaño')

plt.tight_layout()
plt.show()

Esta gráfica de cola larga evidencia que una mínima cantidad de géneros acumulan un gran volúmen de canciones, a su vez, más de 120 géneros de un total de 149 tienen un mínimo de tracks.  

**Conclusión**

La asimetría en la distribución del volumen de canciones por género invalida su uso como criterio de segmentación directa. Priorizar los géneros con mayor volumen de registros dejaría por fuera la gran diversidad del catálogo, ignorando piezas de alto rendimiento. Por el contrario, seleccionar géneros basándose únicamente en sus picos de popularidad promedio omitiría la representatividad de su volumen real. Es necesario utilizar modelos avanzados que faciliten procesar variables acústicas en paralelo que determinen el sello aústico global de las pistas.

**Insight**

El género por si solo no es predictor de popularidad



#### Perfil musical

*Análisis de distribución de las variables acústicas*

In [ ]:
# Lista de parámetros a evaluar
acoustic_parameters = [
    'beats_per_minute_bpm', 'energy', 'danceability', 'loudness_db', 
    'liveness', 'valence', 'length_duration', 'acousticness', 'speechiness'
]

# Configuración de gráficos de distribución
plt.figure(figsize=(18, 14))

for i, col in enumerate(acoustic_parameters):
    plt.subplot(3, 3, i + 1)
    sns.histplot(data=spotify, x=col, kde=True, color='#1DB954', bins=30)
    plt.title(f'Distribución de {col}', fontsize=12, fontweight='bold')
    plt.xlabel('')
    plt.ylabel('')

plt.tight_layout()
plt.show()

# Boxplots para visualizar outliers en variables de escala abierta
plt.figure(figsize=(18, 5))
sub_outliers = ['length_duration', 'beats_per_minute_bpm', 'loudness_db']

for i, col in enumerate(sub_outliers):
    plt.subplot(1, 3, i + 1)
    sns.boxplot(data=spotify, x=col, color='gray', 
                flierprops={"markerfacecolor":"blue", "marker":"o"})
    plt.title(f'Boxplot de Diagnóstico: {col}', fontsize=12, fontweight='bold')
    plt.xlabel('')

plt.tight_layout()
plt.show()
plt.tight_layout()
plt.show()

Las variables beats_per_minute_bpm y danceability tienen distribuciones normales concentradas en 120 BPM y 53.2. energy y valence presentan distribuciones extendidas por todo el catálogo, esta última con sesgo hacia valores bajos. lenght_duration se concentra masivamente entre los 200 y 300 segundos, con cola larga que se extiende hasta los 1400 segundos.
loudness_db esta sesgada a la derecha, la concentración del volumen está entre -10 db y -5 db, acousticness, liveness y speechiness presentan sesgo de valores muy bajos.


length_duration: la mediana de la duración de las canciones es de 245 segundos (4.1 minutos), los diagramas de caja muestran que  posee valores extremadamente atípicos hacia la derecha los cuales superan los 1,000 segundos (16.7 minutos).

beats_per_minute_bpm: a pesar de su distribución normal, la variable presenta outliers por debajo de 50 bpm y por encima de 190 BPM

loudness_db: muestra outliers por debajo de los -18 db y llegan hasta los -27 db.


*Correlación numérica de los atributos acústicos y popularidad*

In [ ]:
# Definición de variables
columns_corr = acoustic_parameters + ['popularity']

# Matriz de correlación
matriz_corr = spotify[columns_corr].corr()

# Mapa de calor
plt.figure(figsize=(10, 4))

sns.heatmap(matriz_corr, cmap='BrBG', vmax=1, vmin=-1, center=0, annot=True, fmt=".2f", linewidths=.5, 
    cbar_kws={"shrink": .8}
)

plt.title('Matriz de Correlación de Atributos acústicos vs. Popularidad', fontsize=14, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right')
plt.show()

La relación entre energy y loudness_db muestra un fuerte coeficiente positivo, en cambio con acousticness la correlación es negativa. Todos los atributos de manera individual muestran correlaciones entre 0.0 y 0.17 frente a la popularidad. 


No se observa relación directa de ninguna variable con la popularidad. Para los modelos de Machine Learning las variables relevantes serían popularity como determinante del éxito comercial, energy (intensidad), danceability (adecuada para bailar), acousticness (acústica) como filtro de contraste para separar lo orgánico de lo digital y valence (positividad). Se descartan speechiness y liveness porque presentan sesgos extremos en su distribución y no aportan información útil para que el algoritmo diferencie una pista de otra. beats_per_minute_bpm y lenght_duration tampoco se tendrán en cuenta por sus escalas abiertas, presencia de outliers y nula contribución a la identificación del aura musical.

**Insight**

El éxito comercial es multifactorial, ninguna variable aislada impulsa el rendimiento por si sola.

### Que música se escucha en todas las épocas?

*Popularidad por década*

In [ ]:
spotify["decade"] = (spotify["year"] // 10) * 10

decade_summary = (
    spotify.groupby("decade")
    .agg(
        total_canciones=("title", "count"),
        popularity_promedio=("popularity", "mean"),
        popularity_mediana=("popularity", "median")
    )
    .reset_index()
    .sort_values("decade")
)

print('Resumen de la popularidad promedio por década:')
decade_summary

# Grafico de popularida por década
plt.figure(figsize=(10, 6))

sns.barplot(
    data=decade_summary,
    x="decade",
    y="popularity_promedio",
    color="gray"
)

plt.title("Popularity promedio por década", fontsize=16, fontweight="bold")
plt.xlabel("Década")
plt.ylabel("Popularity promedio")

for index, row in decade_summary.iterrows():
    plt.text(index, row["popularity_promedio"] + 0.5, f"{row['popularity_promedio']:.1f}", ha="center")

plt.show()

La popularidad promedio sigue un ritmo dececiente a través del tiempo con scores desde 66.2 a 56.9

*Frecuencia de géneros por década*

In [ ]:
# Crea una columna en el dataset con la década
spotify['decade'] = (spotify['year'] // 10) * 10
all_decades = sorted(spotify['decade'].unique())

# Visualiza los 10 géneros con mayor volumen total
genre_inter = spotify['top_genre'].value_counts().head(10).index

# Filtra usando únicamente este grupo
df_filtered_temporal = spotify[spotify['top_genre'].isin(genre_inter)]

# Genera la matriz de frecuencia limpia para la visualización
matrix_frequency = pd.crosstab(
    df_filtered_temporal['top_genre'], 
    df_filtered_temporal['decade']
).reindex(genre_inter) 

print("Matriz de frecuencia de los top 10 géneros más frecuentes por década")
print(matrix_frequency)
print("-" * 60)

plt.figure(figsize=(10, 4))

# Bucle a través de cada género validado para graficar su trayectoria histórica
for genre in genre_inter:
    genre_data = df_filtered_temporal[df_filtered_temporal['top_genre'] == genre].groupby('decade').size()
    genre_data = genre_data.reindex(all_decades, fill_value=0)
    
    plt.plot(genre_data.index, genre_data.values, marker='o', linewidth=2.5, label=genre)

plt.title('Top 10 géneros con mayor volumen en el catálogo', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Década', fontsize=12)
plt.ylabel('Volumen de pistas', fontsize=12)
plt.xticks(all_decades)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title="Top 10")

plt.tight_layout()
plt.show()


Del top 10 de los géneros más escuchados a través del tiempo es adult standards, album rock tuvo su auge entre 1960 y 1990, aún sigue vigente pero decreciendo en frecuencia.

**Insight**

Incluir tracks populares de los géneros más escuchados intergeneracionalmente funciona como ancla de retención que aporta consistencia en las listas premium.

#### ¿Cómo influyen los premios Grammy en las métricas de rendimiento?

#### Exploración dataset grammys

In [ ]:
# carga del dataset

grammy = pd.read_csv("../data/processed/grammy_awards_clean_final.csv")

# Verificación estructura de datos

print('Columnas de spotify:')
print(spotify.columns)

print('Columnas de grammy:')
print(grammy.columns)

In [ ]:
# Verificación tamaño dataset y valores faltantes
print("Tamaño de dataset:", grammy.shape)
print("\n Número de valores ausentes:")
print(grammy.isnull().sum())
print("\n Duplicados:", grammy.duplicated().sum())

# Muestra de datos
display(grammy.head(5))


El dataset no presenta valores nulos, presenta coluumnas normalizadas y los tipos de datos correctos, los valores nulos se tratarán cuando el análisis lo requiera.

*Integración de los dataset*

In [ ]:
# Define un motor de estandarización para eliminar espacios, minúsculas y caracteres especiales.
def crear_llave_match(serie):
    return (
        serie.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

# Mapea las columnas 'title' de Spotify con 'nominee' de los Grammys.
spotify["artist_match"] = crear_llave_match(spotify["artist"])
spotify["title_match"] = crear_llave_match(spotify["title"])

grammy["artist_match"] = crear_llave_match(grammy["artist"])
grammy["title_match"] = crear_llave_match(grammy["nominee"])

# Asegura que el Merge no sufra omisiones o pérdida de registros.
display(spotify[["artist", "title", "artist_match", "title_match"]].head())
display(grammy[["artist", "nominee", "artist_match", "title_match"]].head())

In [ ]:
# Agrupa historial de grammys por categoría y ganadores
grammy_artist_summary = (
    grammy.groupby("artist_match")
    .agg(
        artist_grammy_nominations=("category", "count"),
        artist_grammy_wins=("winner", "sum")
    )
    .reset_index()
)

# Cruza spotify con los premios por llave de artista
spotify_grammy = spotify.merge(
    grammy_artist_summary,
    on="artist_match",
    how="left"
)

# Convierte valores nulos en ceros
spotify_grammy["artist_grammy_nominations"] = spotify_grammy["artist_grammy_nominations"].fillna(0).astype(int)
spotify_grammy["artist_grammy_wins"] = spotify_grammy["artist_grammy_wins"].fillna(0).astype(int)
spotify_grammy["artist_has_grammy_record"] = spotify_grammy["artist_grammy_nominations"] > 0

# Visualización
spotify_grammy[
    ["title", "artist", "popularity", "artist_grammy_nominations", "artist_grammy_wins", "artist_has_grammy_record"]
].head()

In [ ]:
# Compara la popularidad de artitas ganadores y no ganadores de grammy
artist_grammy_comparison = (
    spotify_grammy.groupby("artist_has_grammy_record")
    .agg(
        total_canciones=("title", "count"),
        popularity_promedio=("popularity", "mean"),
        tasa_premium=("popularity", lambda x: (x >= 72).mean() * 100),
        tasa_zona_critica=("popularity", lambda x: (x <= 49).mean() * 100)
    )
    .reset_index()
)

# Matriz comparativa
display(artist_grammy_comparison)


# Boxplot comparativo
plt.figure(figsize=(8, 5))

sns.boxplot(
    data=spotify_grammy,
    x="artist_has_grammy_record",
    y="popularity",
    color='#1DB954'
)

plt.title("Artistas con sin historial Grammy vs con historial Grammy", fontsize=14, fontweight="bold")
plt.xlabel("Artista con historial Grammy")
plt.ylabel("Popularity Score")
plt.xticks([0, 1], ["No", "Sí"])

plt.show()


La popularidad promedio de los artistas ganadores de premios grammy es de 65 puntos, 10 puntos mayor que la de los artistas que no han sido galardonados con el premio. 

**Insight**

Los artistas ganadores de premios grammy gozan de mayor popularidad que los artistas que no han sido premiados, por lo que su presencia en las listas premium podría contribuir en el sello de calidad y prestigio.

*Géneros más populares Vs. reconocimiento Grammy*

In [ ]:
# Agrupa el catálogo por genero para hallar volumen total de pistas y éxito relativo
genre_grammy_summary = (
    spotify_grammy
    .groupby("top_genre")
    .agg(
        total_canciones=("title", "count"),
        popularity_promedio=("popularity", "mean"),
        canciones_artistas_grammy=("artist_has_grammy_record", "sum")
    )
    .reset_index()
)

# Calcula peso relativo porcentual del prestigio dentro del stock de cada género
genre_grammy_summary["porcentaje_grammy"] = (
    genre_grammy_summary["canciones_artistas_grammy"] /
    genre_grammy_summary["total_canciones"] * 100
)

# Filtro de al menos 20 pistas por categoría
top_genre_grammy = (
    genre_grammy_summary
    .query("total_canciones >= 20")
    .sort_values("popularity_promedio", ascending=False)
    .head(10)
)

# Visualización
display(top_genre_grammy)

fig, ax1 = plt.subplots(figsize=(12, 6))

sns.barplot(
    data=top_genre_grammy,
    x="popularity_promedio",
    y="top_genre",
    ax=ax1,
    color="#008080"
)

ax1.set_xlabel("Popularity promedio")
ax1.set_ylabel("Género")
ax1.set_title("Top géneros por Popularity promedio y presencia Grammy", fontsize=16, fontweight="bold")

ax2 = ax1.twiny()

sns.lineplot(
    data=top_genre_grammy,
    x="porcentaje_grammy",
    y="top_genre",
    ax=ax2,
    color="#f39c12",
    marker="o",
    linewidth=2
)

ax2.set_xlabel("% canciones de artistas con historial Grammy")

plt.show()

Los géneros con mayor popularity promedio representan oportunidades claras para playlists premium, si además concentran artistas con historial Grammy el argumento de priorización se vuelve más fuerte porque combinan rendimiento de plataforma y reconocimiento externo como es el caso british invasion y modern rock.

**Insight**

Priorizar géneros que combinen alto Popularity promedio, volumen suficiente de canciones, presencia de artistas con historial Grammy. Estos géneros son candidatos fuertes para playlists premium, campañas promocionales y recomendaciones automáticas.
